# Romania 2001: school results and students arriving from other counties

This notebook acquires **five additional Ministry report families** that we had not collected in the earlier applicant-score notebooks:

1. **School Results View:** the menu’s “admitted + rejected per school” link opens a school directory; each school link opens a list of its applicants, their placement destination and program, or no placement. This view can later be compared with the separate **Gymnasium Applicant View**, which has examination and school-grade components.
2. **Incoming Admitted View:** students from other counties who were admitted to a program in the destination county; the page prints their originating school and destination program.
3. **Incoming Rejected View:** students from other counties who were unassigned in this round; the page prints their originating school.
4. **Incoming Candidate View:** students from other counties before allocation.
5. **Resident Candidate View:** residents of the county before allocation. This is distinct from the already saved general candidate list.

The original county-wide general candidate, placement, unassigned, specialization, and school-directory reports already have their own saved checkpoints; this notebook complements them. The default county list begins with **Bucharest (B)** and then the six counties already used in our pilot (AB, CS, GL, TL, AR, SB). Edit it to add further counties. The worker reads the saved Ministry county menu for each exact URL; it never guesses a report URL. **No acquisition starts until you run Cell 4.** Saved pages live under `~/Desktop/VECTOR_temp`, outside Dropbox and Git. Failed URLs stay unresolved for a later pass. This notebook does not match people, compute transfer rates, or make HERO curves.

Use the `sports_net` Python kernel in Cursor. Run only one Wayback downloader at a time so the pacing and 429 cooldowns have a clear meaning.


## Cell 1 — visible settings

Edit the county list and report families here. For a fast first pass on cross-county arrivals, use `incoming_candidates`, `incoming_admitted`, and `incoming_rejected`; add `school_results` for the longer per-school run. The settings below request all five.

In [ ]:
# Bucharest is first because its 1,317 incoming admitted records could directly
# explain a large part of the discrepancy in our earlier Bucharest source audit.
COUNTIES = ("B", "AB", "CS", "GL", "TL", "AR", "SB")

# These are five DIFFERENT Ministry webpage families, not aliases.
FAMILIES = ("incoming_candidates", "incoming_admitted", "incoming_rejected",
            "resident_candidates", "school_results")

# This cap is per run. Re-running verifies and skips pages already saved.
MAX_NEW_PAGES = 400
MAX_HTTP_ATTEMPTS = 1200
MAX_HOURS = 12.0

# One shared adaptive client; minimum/maximum are the allowed interval bounds.
MIN_INTERVAL_SECONDS = 22.0
MAX_INTERVAL_SECONDS = 50.0

# After a first 429: wait at least 600 seconds; after an immediate second:
# wait at least 1500 seconds. A third consecutive 429 safely stops the run.
FIRST_429_COOLDOWN_SECONDS = 600
SECOND_429_COOLDOWN_SECONDS = 1500
SERVER_ERROR_WAIT_SECONDS = 10.0  # Extra minimum pause after HTTP 500/503.

# Keep this False for a preview, then set True yourself to acquire pages in Cell 4.
RUN_ACQUISITION = False
print("Counties:", ", ".join(COUNTIES))
print("New source families:", ", ".join(FAMILIES))
print("To download, set RUN_ACQUISITION = True and run Cell 4.")


## Cell 2 — load the source-only worker

This imports the Python code. It makes no network requests. If the module was updated while the notebook was open, `reload` picks up the new code.

In [ ]:
import importlib
import sys
from pathlib import Path

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "education_analysis/code").exists():
    REPO = REPO.parent
CODE = REPO / "education_analysis/code"
if not CODE.exists():
    raise FileNotFoundError("Open this notebook from within the new_VECTOR_work repository")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import EDUCATION_20261006_romania_school_results_incoming_acquisition as worker
worker = importlib.reload(worker)
assert set(COUNTIES) <= set(worker.COUNTIES)
assert set(FAMILIES) <= set(worker.FAMILIES)
print("Worker:", worker.__file__)
print("Private page folder:", worker.PAGES)


## Cell 3 — no-network source inventory

The first preview knows only the menu URL for each family. Once a directory or first pagination page is downloaded, its school/page links appear in later previews. “1 URL known” therefore does **not** mean the family has only one page. A projected page total is calculated from the Ministry’s printed total and page range, or from its school-directory links. It is a download target, not proof that every applicant is represented.

In [ ]:
worker.main(["--counties", *COUNTIES, "--families", *FAMILIES])


## Cell 4 — researcher-controlled download

Set `RUN_ACQUISITION = True` in Cell 1, rerun Cell 1, then run this cell. The output announces every request, wait reason, verified download, unresolved URL, and progress count. On a 429, the complete response and exact URL are printed. The worker saves each verified page immediately and can be restarted without discarding it. Stop the cell whenever you need to; the next run resumes.

In [ ]:
if not RUN_ACQUISITION:
    print("Download is OFF. Set RUN_ACQUISITION = True in Cell 1, rerun Cell 1, then rerun this cell.")
else:
    args = [
        "--run", "--counties", *COUNTIES, "--families", *FAMILIES,
        "--max-new-pages", str(MAX_NEW_PAGES),
        "--max-http-attempts", str(MAX_HTTP_ATTEMPTS),
        "--max-hours", str(MAX_HOURS),
        "--min-interval-seconds", str(MIN_INTERVAL_SECONDS),
        "--max-interval-seconds", str(MAX_INTERVAL_SECONDS),
        "--first-429-cooldown-seconds", str(FIRST_429_COOLDOWN_SECONDS),
        "--second-429-cooldown-seconds", str(SECOND_429_COOLDOWN_SECONDS),
        "--server-error-wait-seconds", str(SERVER_ERROR_WAIT_SECONDS),
    ]
    print("Starting source-only acquisition for", ", ".join(COUNTIES), flush=True)
    worker.main(args)


## Cell 5 — readable checkpoint summary

This reads the worker’s private status JSON. It displays the number of verified pages against the Ministry report’s projected page count once the first source page reveals that count. Before that, it says “total not known yet.” It never prints names, scores, or raw HTML. If Cell 4 was already running before this notebook update, let it finish; reload Cell 2 before the next run to get the improved live messages.

In [ ]:
# Reload the worker after an edit, so its saved-page inventory and projections are current.
worker = importlib.reload(worker)
if not worker.STATUS.exists():
    print("No saved status yet. Cell 3 is preview-only; Cell 4 creates status when acquisition runs.")
else:
    worker.main(["--counties", *COUNTIES, "--families", *FAMILIES])
    print("\nThis preview used saved checkpoints only; it made no Wayback request.")


## After acquisition

Completing these downloads does not itself establish who transferred or resolve person matches. The next **offline source check** must compare the school-result pages to the score-bearing applicant pages, reconcile incoming admitted and rejected rows to destination totals, and flag ambiguous or missing records. We will keep every unresolved address open until Charles decides what to do. No national outcome analysis is launched by this notebook.
